# Final token-embedding architecture, sampling, and Close-scale optimisation

This notebook runs the final validation-only token-model programme in four controlled phases:

1. **Six CE-selected fits** using the existing learned hierarchical token embedding.
2. **Architecture screening at temperature 1.0** using ten complete sampled paths. The two lowest-validation-CE models, embedded dynamic ModernTCN, and the existing BSQ-input dynamic ModernTCN control are compared in decoded price space.
3. **Temperature refinement** of the sampled architecture winner at `0.8`, `1.0`, and `1.2`.
4. **Close scale/volatility ablation**: the selected architecture is retrained once with causal `log(mean Close)` and `log(std Close / mean Close)` features, then evaluated at the fixed selected temperature.

All model checkpoints are selected by validation coarse-token cross-entropy. Temperature and architecture screening use September validation only. The proposed-model held-out test split is not accessed.

All ten decoded Close-price paths are retained for every stochastic policy.

In [ ]:
## Cell 1 — imports

from importlib import metadata
from pathlib import Path
from time import perf_counter

import json
import os
import shlex
import shutil
import subprocess
import sys

import numpy as np
import pandas as pd
import torch
import yaml

from IPython.display import display
from google.colab import drive, userdata


In [ ]:
## Cell 2 — mount Google Drive

drive.mount("/content/drive")


In [ ]:
## Cell 3 — paths and locked experiment settings

REPO_URL = "https://github.com/VishR-94/dynamic_graphs_thesis.git"
BRANCH = "main"
REPO_NAME = REPO_URL.removesuffix(".git").rstrip("/").split("/")[-1]
REPO_DIR = Path("/content") / REPO_NAME

DATA_DIR = Path(
    "/content/drive/Shareddrives/Vishal/data/cached_datasets/"
    "exp-1m-95s-24y/session"
)
TOKEN_DIR = Path(
    "/content/drive/MyDrive/dissertation/final_model/tokens"
)
TRAIN_CACHE = TOKEN_DIR / "origin_aligned_train_tokens.pt"
VALIDATION_CACHE = TOKEN_DIR / "origin_aligned_val_tokens.pt"

OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/dissertation/final_model/tokenized_model_final"
)
EXPERIMENT_CONTROL_DIR = OUTPUT_ROOT / "_experiment_control"

# Existing completed BSQ-input control. It remains eligible to win.
BSQ_CONTROL_ROOT = Path(
    "/content/drive/MyDrive/dissertation/final_model/"
    "tokenized_modern_tcn_final"
)

DYNAMIC_CONFIG = REPO_DIR / "configs/dynamic_graph.yaml"
FORECASTING_CONFIG = REPO_DIR / "configs/forecasting.yaml"

MAX_EPOCHS = 100
PATIENCE = 10
TRAIN_BATCH_SIZE = 2
VALIDATION_BATCH_SIZE = 2
NUM_WORKERS = 0
VALIDATION_DECODE_EVERY = 0
DECODE_SERIES_BATCH_SIZE = 64
SEED = 42

SCREENING_TEMPERATURE = 1.0
TEMPERATURE_GRID = (0.8, 1.0, 1.2)
SAMPLE_COUNT = 10
TOP_K = 0
TOP_P = 0.9
SAMPLING_SEED = 42

REQUEST_WANDB = True
WANDB_PROJECT = "dynamic-graph-financial-forecasting"
WANDB_ENTITY = None
FORCE_OVERWRITE = False

EXPECTED_TRAIN_WINDOWS = 3173
EXPECTED_VALIDATION_WINDOWS = 380
EXPECTED_ASSETS = 93
EXPECTED_CONTEXT_LENGTH = 60
EXPECTED_PREDICTION_LENGTH = 60
EXPECTED_EVALUATION_HORIZONS = (1, 5, 15, 30, 60)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
EXPERIMENT_CONTROL_DIR.mkdir(parents=True, exist_ok=True)

print("Repository:", REPO_DIR)
print("Data:", DATA_DIR)
print("Token caches:", TOKEN_DIR)
print("New experiment output:", OUTPUT_ROOT)
print("Existing BSQ control root:", BSQ_CONTROL_ROOT)
print("Screening temperature:", SCREENING_TEMPERATURE)
print("Temperature comparison:", TEMPERATURE_GRID)


In [ ]:
## Cell 4 — authenticate, clone/update, and initialise submodules

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

GITHUB_TOKEN = secret("GITHUB_TOKEN")
if not GITHUB_TOKEN:
    raise RuntimeError("Add GITHUB_TOKEN to Colab Secrets.")

askpass = Path("/tmp/git-askpass.sh")
askpass.write_text(
    "#!/bin/sh\ncase \"$1\" in\n"
    "  *Username*) echo \"x-access-token\" ;;;\n"
    "  *Password*) echo \"$GITHUB_TOKEN\" ;;;\n"
    "esac\n".replace(";;;", ";;"),
    encoding="utf-8",
)
askpass.chmod(0o700)

git_env = os.environ.copy()
git_env.update(
    {
        "GITHUB_TOKEN": GITHUB_TOKEN,
        "GIT_ASKPASS": str(askpass),
        "GIT_ASKPASS_REQUIRE": "force",
        "GIT_TERMINAL_PROMPT": "0",
    }
)

HF_TOKEN = secret("HF_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("Hugging Face authentication enabled.")
else:
    print("No HF_TOKEN found; public downloads are unauthenticated.")

WANDB_API_KEY = secret("WANDB_API_KEY")
if REQUEST_WANDB and WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
    ACTIVE_WANDB_MODE = "online"
else:
    ACTIVE_WANDB_MODE = "disabled"

if REPO_DIR.exists() and not (REPO_DIR / ".git").is_dir():
    shutil.rmtree(REPO_DIR)

if (REPO_DIR / ".git").is_dir():
    status_lines = subprocess.check_output(
        ["git", "status", "--short"],
        cwd=REPO_DIR,
        text=True,
    ).splitlines()

    # Recover cleanly from an interrupted manual submodule clone in Colab.
    allowed_untracked = {
        "?? external/ModernTCN/",
        "?? external/Kronos/",
        "?? external/BaseDyGraph/",
    }
    unexpected = [line for line in status_lines if line not in allowed_untracked]
    if unexpected:
        raise RuntimeError(
            "Colab repository has uncommitted changes:\n"
            + "\n".join(unexpected)
        )
    for line in status_lines:
        if line in allowed_untracked:
            shutil.rmtree(REPO_DIR / line.removeprefix("?? ").rstrip("/"))

    subprocess.run(
        ["git", "-C", str(REPO_DIR), "fetch", "origin"],
        check=True,
        env=git_env,
    )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "checkout", BRANCH],
        check=True,
        env=git_env,
    )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH],
        check=True,
        env=git_env,
    )
else:
    subprocess.run(
        [
            "git",
            "clone",
            "--recurse-submodules",
            "--branch",
            BRANCH,
            REPO_URL,
            str(REPO_DIR),
        ],
        check=True,
        env=git_env,
    )

subprocess.run(
    ["git", "-C", str(REPO_DIR), "submodule", "sync", "--recursive"],
    check=True,
    env=git_env,
)
subprocess.run(
    ["git", "-C", str(REPO_DIR), "submodule", "update", "--init", "--recursive"],
    check=True,
    env=git_env,
)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

GIT_COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True
).strip()

for submodule_name, hash_file in (
    ("ModernTCN", "modern_tcn_commit_hash.md"),
    ("Kronos", "kronos_commit_hash.md"),
):
    submodule_dir = REPO_DIR / "external" / submodule_name
    pinned = (REPO_DIR / "docs" / hash_file).read_text(encoding="utf-8").strip()
    observed = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=submodule_dir, text=True
    ).strip()
    if observed != pinned:
        raise AssertionError(
            f"{submodule_name} revision differs: expected {pinned}, observed {observed}."
        )
    print(f"{submodule_name} commit:", observed)

print("Project commit:", GIT_COMMIT)
subprocess.run(["git", "submodule", "status"], cwd=REPO_DIR, check=True)


In [ ]:
## Cell 5 — install and verify dependencies

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "-r",
        str(REPO_DIR / "requirements.txt"),
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--upgrade",
        "transformers==5.3.0",
        "huggingface_hub==1.3.0",
    ],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "--yes", "diffusers"],
    check=True,
)
pip_check = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    text=True,
    capture_output=True,
)
if pip_check.stdout:
    print(pip_check.stdout)
if pip_check.stderr:
    print(pip_check.stderr)
if pip_check.returncode:
    raise RuntimeError("Dependency verification failed.")
print("Dependencies installed successfully.")


In [ ]:
## Cell 6 — verify code, caches, and contracts before using the GPU

from src.data.token_graph_dataset import load_origin_aligned_token_cache

required_paths = [
    REPO_DIR / "src/models/dynamic_graph/modern_tcn_token.py",
    REPO_DIR / "src/models/dynamic_graph/model.py",
    REPO_DIR / "src/training/run_dynamic_graph.py",
    REPO_DIR / "src/training/token_embedding_final_sweep.py",
    REPO_DIR / "scripts/test_tokenized_modern_tcn.py",
    REPO_DIR / "scripts/test_token_embedding_final_sweep.py",
    REPO_DIR / "external/ModernTCN/ModernTCN-Long-term-forecasting/models/ModernTCN.py",
    REPO_DIR / "external/Kronos/model/kronos.py",
    DYNAMIC_CONFIG,
    FORECASTING_CONFIG,
    DATA_DIR / "train.pt",
    DATA_DIR / "val.pt",
    DATA_DIR / "test.pt",
    TRAIN_CACHE,
    VALIDATION_CACHE,
]
for path in required_paths:
    if not path.is_file():
        raise FileNotFoundError(path)
if not torch.cuda.is_available():
    raise RuntimeError("Select a CUDA runtime before running this notebook.")

subprocess.run(
    [
        sys.executable,
        "-m",
        "py_compile",
        str(REPO_DIR / "src/models/dynamic_graph/contracts.py"),
        str(REPO_DIR / "src/models/dynamic_graph/modules.py"),
        str(REPO_DIR / "src/models/dynamic_graph/modern_tcn_token.py"),
        str(REPO_DIR / "src/models/dynamic_graph/model.py"),
        str(REPO_DIR / "src/training/run_dynamic_graph.py"),
        str(REPO_DIR / "src/training/token_embedding_final_sweep.py"),
        str(REPO_DIR / "scripts/test_tokenized_modern_tcn.py"),
        str(REPO_DIR / "scripts/test_token_embedding_final_sweep.py"),
    ],
    check=True,
    cwd=REPO_DIR,
)
for module in (
    "src.models.dynamic_graph.config",
    "scripts.test_tokenized_modern_tcn",
    "scripts.test_token_embedding_final_sweep",
):
    subprocess.run(
        [sys.executable, "-m", module],
        check=True,
        cwd=REPO_DIR,
    )

train_cache = load_origin_aligned_token_cache(TRAIN_CACHE)
validation_cache = load_origin_aligned_token_cache(VALIDATION_CACHE)

for label, cache, expected_windows in (
    ("train", train_cache, EXPECTED_TRAIN_WINDOWS),
    ("validation", validation_cache, EXPECTED_VALIDATION_WINDOWS),
):
    expected_context = (
        expected_windows,
        EXPECTED_CONTEXT_LENGTH,
        EXPECTED_ASSETS,
        2,
    )
    expected_target = (
        expected_windows,
        EXPECTED_PREDICTION_LENGTH,
        EXPECTED_ASSETS,
    )
    if tuple(cache["context_tokens"].shape) != expected_context:
        raise AssertionError(
            f"{label} context shape differs: {tuple(cache['context_tokens'].shape)}"
        )
    if tuple(cache["target_s1"].shape) != expected_target:
        raise AssertionError(
            f"{label} target_s1 shape differs: {tuple(cache['target_s1'].shape)}"
        )
    if str(cache.get("s1_id_space", "kronos_original")) != "kronos_original":
        raise AssertionError(f"{label} cache is not in original Kronos ID space.")
    if int(cache.get("s1_vocabulary_size", 1024)) != 1024:
        raise AssertionError(f"{label} cache is not the full 1024-class cache.")
    observed_horizons = tuple(int(v) for v in cache["evaluation_horizons"])
    if observed_horizons != EXPECTED_EVALUATION_HORIZONS:
        raise AssertionError(
            f"{label} horizons differ: {observed_horizons}."
        )
    for tensor_name, values in (
        ("context s1", torch.as_tensor(cache["context_tokens"][..., 0])),
        ("context s2", torch.as_tensor(cache["context_tokens"][..., 1])),
        ("target s1", torch.as_tensor(cache["target_s1"])),
    ):
        minimum = int(values.min().item())
        maximum = int(values.max().item())
        if minimum < 0 or maximum >= 1024:
            raise AssertionError(
                f"{label} {tensor_name} range is [{minimum}, {maximum}]."
            )
    print(f"{label.capitalize()} cache validated:", expected_context)

print("GPU:", torch.cuda.get_device_name(0))
print("Torch:", metadata.version("torch"))
print("All final-experiment contracts passed.")

del train_cache
del validation_cache


In [ ]:
## Cell 7 — import experiment helpers and define resumable launch utilities

from src.training.token_embedding_final_sweep import (
    BSQ_CONTROL_RUN_NAME,
    TokenArchitectureSpec,
    clone_with_close_scale_features,
    load_selected_validation_ce,
    load_temperature_result,
    make_architecture_specs,
    make_bsq_control_spec,
    save_specs,
    select_screening_specs,
    summarise_validation_ce,
    temperature_label,
)


def load_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def run_root(spec):
    return BSQ_CONTROL_ROOT if spec.run_name == BSQ_CONTROL_RUN_NAME else OUTPUT_ROOT


def run_dir(spec):
    return run_root(spec) / spec.run_name


def run_is_complete(spec):
    directory = run_dir(spec)
    metadata_path = directory / "run_metadata.json"
    checkpoint_path = directory / "best_checkpoint.pt"
    return (
        metadata_path.is_file()
        and checkpoint_path.is_file()
        and load_json(metadata_path).get("status") == "completed"
    )


def common_command(spec, *, wandb_mode):
    command = [
        sys.executable,
        "-u",
        "-m",
        "src.training.run_dynamic_graph",
        "--dynamic-config",
        str(DYNAMIC_CONFIG),
        "--forecasting-config",
        str(FORECASTING_CONFIG),
        "--train-cache",
        str(TRAIN_CACHE),
        "--val-cache",
        str(VALIDATION_CACHE),
        "--data-dir",
        str(DATA_DIR),
        "--output-dir",
        str(run_root(spec)),
        "--run-name",
        spec.run_name,
        "--preset",
        spec.preset,
        "--device",
        "cuda",
        "--max-epochs",
        str(MAX_EPOCHS),
        "--patience",
        str(PATIENCE),
        "--train-batch-size",
        str(TRAIN_BATCH_SIZE),
        "--validation-batch-size",
        str(VALIDATION_BATCH_SIZE),
        "--num-workers",
        str(NUM_WORKERS),
        "--decode-series-batch-size",
        str(DECODE_SERIES_BATCH_SIZE),
        "--validation-decode-every",
        str(VALIDATION_DECODE_EVERY),
        "--wandb-mode",
        wandb_mode,
        "--wandb-project",
        WANDB_PROJECT,
        "--wandb-tags",
        "token-embedding-final",
        spec.temporal_family,
        spec.graph_type,
        spec.token_input_representation,
        "coarse-only",
        "ce-selection",
        "--mixed-precision",
    ]
    if WANDB_ENTITY is not None:
        command.extend(["--wandb-entity", WANDB_ENTITY])
    for override in spec.overrides:
        command.extend(["--set", override])
    return command


def build_training_command(spec):
    command = common_command(spec, wandb_mode=ACTIVE_WANDB_MODE)
    directory = run_dir(spec)
    if FORCE_OVERWRITE:
        command.append("--overwrite")
        return command
    if run_is_complete(spec):
        return None
    if (directory / "last_checkpoint.pt").is_file():
        command.append("--resume")
    elif directory.exists() and any(directory.iterdir()):
        raise RuntimeError(
            "Non-empty, non-resumable run directory:\n"
            f"{directory}\nInspect or remove it before continuing."
        )
    return command


def build_temperature_command(spec, temperatures):
    if not run_is_complete(spec):
        raise RuntimeError(f"Cannot sample an incomplete run: {run_dir(spec)}")
    temperature_values = [float(value) for value in temperatures]
    command = common_command(spec, wandb_mode="disabled")
    command.append("--temperature-sweep")
    for override in (
        f"temperature_sweep.temperatures={temperature_values}",
        f"temperature_sweep.sample_count={SAMPLE_COUNT}",
        f"temperature_sweep.top_k={TOP_K}",
        f"temperature_sweep.top_p={TOP_P}",
        f"temperature_sweep.seed={SAMPLING_SEED}",
    ):
        command.extend(["--set", override])
    return command


def stream_command(command, *, label):
    print("=" * 88)
    print(label)
    print(" ".join(shlex.quote(value) for value in command))
    print("=" * 88)
    start = perf_counter()
    process = subprocess.Popen(
        command,
        cwd=REPO_DIR,
        env=os.environ.copy(),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    if process.stdout is None:
        raise RuntimeError("Could not capture subprocess output.")
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)
    print(f"Finished in {(perf_counter() - start) / 3600:.2f} hours.")


def launch_training(spec):
    command = build_training_command(spec)
    if command is None:
        print(spec.run_name, "is complete; skipping training.")
        return
    stream_command(command, label=f"Training: {spec.label}")


def launch_temperature(spec, temperatures):
    stream_command(
        build_temperature_command(spec, temperatures),
        label=(
            f"Sampling: {spec.label}; temperatures="
            f"{tuple(float(v) for v in temperatures)}"
        ),
    )


def load_policy_metric_table(spec, temperature):
    record = load_temperature_result(
        run_dir(spec),
        temperature=float(temperature),
        sample_count=SAMPLE_COUNT,
        top_p=TOP_P,
        top_k=TOP_K,
        seed=SAMPLING_SEED,
    )
    table = pd.read_csv(record["Metric table"])
    table["Run"] = spec.run_name
    table["Label"] = spec.label
    table["Temperature"] = float(temperature)
    return record, table


def policy_summary_row(spec, temperature, validation_ce=None):
    record, table = load_policy_metric_table(spec, temperature)
    selected = table.loc[
        (table["metric"] == "cumulative_log_change_mae")
        & table["channel"].astype(str).str.lower().eq("close")
    ].copy()
    selected["horizon"] = pd.to_numeric(selected["horizon"]).astype(int)
    by_horizon = {
        int(row.horizon): float(row.value)
        for row in selected.itertuples(index=False)
    }
    if set(by_horizon) != set(EXPECTED_EVALUATION_HORIZONS):
        raise AssertionError(
            f"Incomplete Log-MAE horizons for {spec.run_name} at T={temperature}."
        )
    return {
        "Run": spec.run_name,
        "Label": spec.label,
        "Token input": spec.token_input_representation,
        "Temporal family": spec.temporal_family,
        "Graph": spec.graph_type,
        "ST blocks": spec.num_st_blocks,
        "Best validation CE": validation_ce,
        "Temperature": float(temperature),
        "Mean Log MAE": float(np.mean(list(by_horizon.values()))),
        **{
            f"Log MAE — {horizon} min": by_horizon[horizon]
            for horizon in EXPECTED_EVALUATION_HORIZONS
        },
        "Generation seconds": record.get("Generation seconds"),
        "Generated s1 accuracy": record.get("Generated s1 accuracy"),
        "Invalid ensemble candles (%)": record.get(
            "Invalid ensemble candles (%)"
        ),
        "Invalid sampled-path candles (%)": record.get(
            "Invalid sampled-path candles (%)"
        ),
        "Sampled path file": record["Sampled path file"],
        "Metric table": record["Metric table"],
    }


def style_summary(frame, caption):
    formats = {
        "Best validation CE": "{:.6f}",
        "Temperature": "{:.2f}",
        "Mean Log MAE": "{:.8f}",
        "Generation seconds": "{:.1f}",
        "Generated s1 accuracy": "{:.6f}",
        "Invalid ensemble candles (%)": "{:.3f}",
        "Invalid sampled-path candles (%)": "{:.3f}",
    }
    for horizon in EXPECTED_EVALUATION_HORIZONS:
        formats[f"Log MAE — {horizon} min"] = "{:.8f}"
    return frame.style.format(formats, na_rep="—").set_caption(caption)


## Phase 1 — train the six learned-embedding architectures

The ModernTCN pair uses the exact existing `D=32`, one-block, patch-8/stride-4, large-kernel-15 backbone. The only representation change relative to the completed BSQ control is the learned hierarchical token/node/position embedding before ModernTCN.

The Transformer variants use `D=96`, eight attention heads, and either one or three **true interlaced ST blocks**. Every model retains the separate one-layer structured-parallel future-query Transformer and predicts all 60 future coarse tokens.

In [ ]:
## Cell 8 — build and inspect the six architecture specifications

ARCHITECTURE_SPECS = make_architecture_specs()
BSQ_CONTROL_SPEC = make_bsq_control_spec()

save_specs(
    EXPERIMENT_CONTROL_DIR / "six_architecture_specs.json",
    ARCHITECTURE_SPECS,
)

spec_table = pd.DataFrame([spec.to_dict() for spec in ARCHITECTURE_SPECS])[
    [
        "run_name",
        "label",
        "temporal_family",
        "graph_type",
        "num_st_blocks",
        "token_input_representation",
        "preset",
    ]
]
display(spec_table.style.set_caption("Locked six-model architecture grid"))
print("Historical BSQ control:", BSQ_CONTROL_SPEC.run_name)
print("Control directory:", run_dir(BSQ_CONTROL_SPEC))


In [ ]:
## Cell 9 — train all six models and rank their CE-selected checkpoints

for index, spec in enumerate(ARCHITECTURE_SPECS, start=1):
    print(f"\nArchitecture fit {index}/{len(ARCHITECTURE_SPECS)}")
    launch_training(spec)

ce_results = summarise_validation_ce(
    OUTPUT_ROOT,
    ARCHITECTURE_SPECS,
    require_all=True,
)
ce_results.to_csv(
    EXPERIMENT_CONTROL_DIR / "architecture_ce_results.csv",
    index=False,
)

display(
    ce_results.style.format(
        {
            "Best validation CE": "{:.6f}",
            "Validation s1 accuracy": "{:.6f}",
            "Spatial beta": "{:.4f}",
            "Graph mean row entropy": "{:.4f}",
            "Graph effective neighbours": "{:.2f}",
        },
        na_rep="—",
    ).set_caption(
        "Six learned-embedding models ranked by lowest September validation CE"
    )
)


## Phase 2 — sampled architecture screening at temperature 1.0

The candidates are the two lowest-CE embedded models, embedded dynamic ModernTCN if it is not already among them, and the completed BSQ-input dynamic ModernTCN control. All candidates use the same ten-path sampling protocol and random seed. The winner is selected by decoded five-horizon mean Log MAE, not CE.

In [ ]:
## Cell 10 — select the screening candidates

if not run_is_complete(BSQ_CONTROL_SPEC):
    raise FileNotFoundError(
        "The completed BSQ control was not found:\n"
        f"{run_dir(BSQ_CONTROL_SPEC)}"
    )

embedded_screening_specs = select_screening_specs(
    ce_results,
    ARCHITECTURE_SPECS,
)

screening_lookup = {}
for spec in (*embedded_screening_specs, BSQ_CONTROL_SPEC):
    screening_lookup.setdefault(spec.run_name, spec)
SCREENING_SPECS = tuple(screening_lookup.values())

screening_selection = {
    "selection_basis": (
        "top two embedded models by validation CE, plus embedded dynamic "
        "ModernTCN if absent, plus the existing BSQ dynamic ModernTCN control"
    ),
    "screening_temperature": SCREENING_TEMPERATURE,
    "sample_count": SAMPLE_COUNT,
    "top_p": TOP_P,
    "top_k": TOP_K,
    "sampling_seed": SAMPLING_SEED,
    "candidate_runs": [spec.to_dict() for spec in SCREENING_SPECS],
}
(EXPERIMENT_CONTROL_DIR / "screening_candidates.json").write_text(
    json.dumps(screening_selection, indent=2, sort_keys=True),
    encoding="utf-8",
)

candidate_rows = []
for spec in SCREENING_SPECS:
    selected_ce = load_selected_validation_ce(run_dir(spec))
    candidate_rows.append(
        {
            "Run": spec.run_name,
            "Label": spec.label,
            "Token input": spec.token_input_representation,
            "Temporal family": spec.temporal_family,
            "Graph": spec.graph_type,
            "ST blocks": spec.num_st_blocks,
            **selected_ce,
        }
    )
display(
    pd.DataFrame(candidate_rows)
    .sort_values(["Best validation CE", "Run"])
    .style.format(
        {
            "Best validation CE": "{:.6f}",
            "Validation s1 accuracy": "{:.6f}",
            "Spatial beta": "{:.4f}",
        },
        na_rep="—",
    )
    .set_caption("Models carried into ten-path architecture screening")
)


In [ ]:
## Cell 11 — run ten-path screening at T=1.0 and select the architecture

for index, spec in enumerate(SCREENING_SPECS, start=1):
    print(f"\nScreening candidate {index}/{len(SCREENING_SPECS)}")
    launch_temperature(spec, (SCREENING_TEMPERATURE,))

screening_rows = []
for spec in SCREENING_SPECS:
    selected_ce = load_selected_validation_ce(run_dir(spec))["Best validation CE"]
    screening_rows.append(
        policy_summary_row(
            spec,
            SCREENING_TEMPERATURE,
            validation_ce=selected_ce,
        )
    )

screening_results = (
    pd.DataFrame(screening_rows)
    .sort_values(["Mean Log MAE", "Run"])
    .reset_index(drop=True)
)
screening_results.to_csv(
    EXPERIMENT_CONTROL_DIR / "architecture_screening_results.csv",
    index=False,
)

display(
    style_summary(
        screening_results,
        "Ten-path architecture screening at temperature 1.0",
    )
)

ARCHITECTURE_WINNER_RUN = str(screening_results.iloc[0]["Run"])
ARCHITECTURE_WINNER_SPEC = screening_lookup[ARCHITECTURE_WINNER_RUN]
architecture_selection = {
    "selected_run": ARCHITECTURE_WINNER_SPEC.run_name,
    "selected_label": ARCHITECTURE_WINNER_SPEC.label,
    "selected_run_dir": str(run_dir(ARCHITECTURE_WINNER_SPEC)),
    "screening_temperature": SCREENING_TEMPERATURE,
    "mean_log_mae": float(screening_results.iloc[0]["Mean Log MAE"]),
    "selection_metric": (
        "strict lowest five-horizon September validation cumulative-log-change MAE"
    ),
    "spec": ARCHITECTURE_WINNER_SPEC.to_dict(),
}
(EXPERIMENT_CONTROL_DIR / "architecture_selection.json").write_text(
    json.dumps(architecture_selection, indent=2, sort_keys=True),
    encoding="utf-8",
)

print("\nSELECTED ARCHITECTURE")
print("Run:", ARCHITECTURE_WINNER_SPEC.run_name)
print("Directory:", run_dir(ARCHITECTURE_WINNER_SPEC))
print("Mean Log MAE:", architecture_selection["mean_log_mae"])


## Phase 3 — temperature refinement

The architecture winner already has a saved `T=1.0` result. The inference runner is policy-resumable, so requesting `(0.8, 1.0, 1.2)` reuses `T=1.0` and any other completed policy rather than decoding it again.

In [ ]:
## Cell 12 — evaluate T=0.8, 1.0, and 1.2 and select temperature

launch_temperature(ARCHITECTURE_WINNER_SPEC, TEMPERATURE_GRID)

temperature_rows = [
    policy_summary_row(
        ARCHITECTURE_WINNER_SPEC,
        temperature,
        validation_ce=load_selected_validation_ce(
            run_dir(ARCHITECTURE_WINNER_SPEC)
        )["Best validation CE"],
    )
    for temperature in TEMPERATURE_GRID
]
temperature_results = (
    pd.DataFrame(temperature_rows)
    .sort_values(["Mean Log MAE", "Temperature"])
    .reset_index(drop=True)
)
temperature_results.to_csv(
    EXPERIMENT_CONTROL_DIR / "temperature_refinement_results.csv",
    index=False,
)

display(
    style_summary(
        temperature_results,
        "Selected architecture — temperature refinement",
    )
)

SELECTED_TEMPERATURE = float(temperature_results.iloc[0]["Temperature"])
temperature_selection = {
    "architecture_run": ARCHITECTURE_WINNER_SPEC.run_name,
    "architecture_run_dir": str(run_dir(ARCHITECTURE_WINNER_SPEC)),
    "selected_temperature": SELECTED_TEMPERATURE,
    "mean_log_mae": float(temperature_results.iloc[0]["Mean Log MAE"]),
    "compared_temperatures": [float(value) for value in TEMPERATURE_GRID],
    "sample_count": SAMPLE_COUNT,
    "top_p": TOP_P,
    "top_k": TOP_K,
    "sampling_seed": SAMPLING_SEED,
}
(EXPERIMENT_CONTROL_DIR / "temperature_selection.json").write_text(
    json.dumps(temperature_selection, indent=2, sort_keys=True),
    encoding="utf-8",
)
print("Selected temperature:", SELECTED_TEMPERATURE)


In [ ]:
## Cell 13 — complete metric table for all three temperatures

metric_frames = []
for temperature in TEMPERATURE_GRID:
    _, table = load_policy_metric_table(
        ARCHITECTURE_WINNER_SPEC,
        temperature,
    )
    metric_frames.append(table)

all_temperature_metrics = pd.concat(metric_frames, ignore_index=True)
close_metrics = all_temperature_metrics.loc[
    all_temperature_metrics["channel"].astype(str).str.lower().eq("close")
].copy()
metric_wide = (
    close_metrics.pivot_table(
        index=["Temperature", "horizon"],
        columns="metric",
        values="value",
        aggfunc="first",
    )
    .sort_index()
)
display(
    metric_wide.style.format("{:.8f}", na_rep="—").set_caption(
        "Complete Close-channel metrics for the selected architecture"
    )
)


## Phase 4 — matched Close scale/volatility ablation

The selected architecture is retrained from scratch with exactly two additional causal features per asset/window:

- `log(context mean Close)` — price level removed by tokenizer normalisation;
- `log(context std Close / context mean Close)` — relative within-window volatility.

Their standardisation statistics are fitted from training windows only. The selected temperature is held fixed, so the comparison changes only the scale-feature input.

In [ ]:
## Cell 14 — build and train the matched scale-feature model

scale_run_name = (
    "scale_"
    + ARCHITECTURE_WINNER_SPEC.token_input_representation.replace(
        "hierarchical_embedding", "embed"
    ).replace("bsq_bits", "bsq")
    + "_"
    + ARCHITECTURE_WINNER_SPEC.run_name
    + "_close_level_vol_ce"
)
SCALE_SPEC = clone_with_close_scale_features(
    ARCHITECTURE_WINNER_SPEC,
    run_name=scale_run_name,
)
save_specs(
    EXPERIMENT_CONTROL_DIR / "scale_feature_spec.json",
    (SCALE_SPEC,),
)
print("Scale run:", SCALE_SPEC.run_name)
print("Preset:", SCALE_SPEC.preset)
print("Source architecture:", ARCHITECTURE_WINNER_SPEC.run_name)
launch_training(SCALE_SPEC)

scale_ce = load_selected_validation_ce(run_dir(SCALE_SPEC))
source_ce = load_selected_validation_ce(run_dir(ARCHITECTURE_WINNER_SPEC))
display(
    pd.DataFrame(
        [
            {
                "Model": "Selected architecture — no scale feature",
                "Run": ARCHITECTURE_WINNER_SPEC.run_name,
                **source_ce,
            },
            {
                "Model": "Matched architecture — Close scale features",
                "Run": SCALE_SPEC.run_name,
                **scale_ce,
            },
        ]
    ).style.format(
        {
            "Best validation CE": "{:.6f}",
            "Validation s1 accuracy": "{:.6f}",
            "Spatial beta": "{:.4f}",
        },
        na_rep="—",
    ).set_caption("CE-selected checkpoints for the scale ablation")
)


In [ ]:
## Cell 15 — evaluate scale model at the fixed selected temperature

launch_temperature(SCALE_SPEC, (SELECTED_TEMPERATURE,))

no_scale_row = policy_summary_row(
    ARCHITECTURE_WINNER_SPEC,
    SELECTED_TEMPERATURE,
    validation_ce=source_ce["Best validation CE"],
)
scale_row = policy_summary_row(
    SCALE_SPEC,
    SELECTED_TEMPERATURE,
    validation_ce=scale_ce["Best validation CE"],
)
scale_comparison = (
    pd.DataFrame([no_scale_row, scale_row])
    .sort_values(["Mean Log MAE", "Run"])
    .reset_index(drop=True)
)
scale_comparison.to_csv(
    EXPERIMENT_CONTROL_DIR / "scale_feature_comparison.csv",
    index=False,
)
display(
    style_summary(
        scale_comparison,
        (
            "Matched Close scale/volatility ablation at fixed temperature "
            f"{SELECTED_TEMPERATURE:g}"
        ),
    )
)

FINAL_RUN_NAME = str(scale_comparison.iloc[0]["Run"])
FINAL_SPEC = (
    SCALE_SPEC
    if FINAL_RUN_NAME == SCALE_SPEC.run_name
    else ARCHITECTURE_WINNER_SPEC
)
FINAL_RUN_DIR = run_dir(FINAL_SPEC)
FINAL_USES_SCALE = FINAL_SPEC.run_name == SCALE_SPEC.run_name
FINAL_MEAN_LOG_MAE = float(scale_comparison.iloc[0]["Mean Log MAE"])

print("\nFINAL VALIDATION-SELECTED TOKEN MODEL")
print("Run:", FINAL_SPEC.run_name)
print("Directory:", FINAL_RUN_DIR)
print("Temperature:", SELECTED_TEMPERATURE)
print("Uses Close scale features:", FINAL_USES_SCALE)
print("Mean Log MAE:", FINAL_MEAN_LOG_MAE)


In [ ]:
## Cell 16 — final full metrics, sampled-path checks, and manifest

final_metric_frames = []
for label, spec in (
    ("No scale feature", ARCHITECTURE_WINNER_SPEC),
    ("Close scale + volatility", SCALE_SPEC),
):
    _, table = load_policy_metric_table(spec, SELECTED_TEMPERATURE)
    table["Scale ablation"] = label
    final_metric_frames.append(table)

final_metrics = pd.concat(final_metric_frames, ignore_index=True)
final_close = final_metrics.loc[
    final_metrics["channel"].astype(str).str.lower().eq("close")
].copy()
final_wide = final_close.pivot_table(
    index=["Scale ablation", "horizon"],
    columns="metric",
    values="value",
    aggfunc="first",
).sort_index()
display(
    final_wide.style.format("{:.8f}", na_rep="—").set_caption(
        "Complete final scale-ablation metrics at the selected temperature"
    )
)

path_rows = []
for stage, spec, temperature in (
    *[("screening", spec, SCREENING_TEMPERATURE) for spec in SCREENING_SPECS],
    *[("temperature_refinement", ARCHITECTURE_WINNER_SPEC, t) for t in TEMPERATURE_GRID],
    ("scale_ablation", SCALE_SPEC, SELECTED_TEMPERATURE),
):
    record = load_temperature_result(
        run_dir(spec),
        temperature=float(temperature),
        sample_count=SAMPLE_COUNT,
        top_p=TOP_P,
        top_k=TOP_K,
        seed=SAMPLING_SEED,
    )
    path = Path(record["Sampled path file"])
    payload = torch.load(path, map_location="cpu", weights_only=False)
    artifacts = payload["sampled_price_path_artifacts"]
    sampled = torch.as_tensor(artifacts["sampled_close_paths"])
    sampled_eval = torch.as_tensor(
        artifacts["sampled_close_paths_at_evaluation_horizons"]
    )
    if int(sampled.shape[0]) != SAMPLE_COUNT:
        raise AssertionError(f"{path} contains {sampled.shape[0]} paths.")
    if tuple(sampled.shape[1:]) != (
        EXPECTED_VALIDATION_WINDOWS,
        EXPECTED_PREDICTION_LENGTH,
        EXPECTED_ASSETS,
        1,
    ):
        raise AssertionError(
            f"Unexpected dense sampled-path shape in {path}: {tuple(sampled.shape)}"
        )
    if tuple(sampled_eval.shape[1:]) != (
        EXPECTED_VALIDATION_WINDOWS,
        len(EXPECTED_EVALUATION_HORIZONS),
        EXPECTED_ASSETS,
        1,
    ):
        raise AssertionError(
            f"Unexpected evaluation sampled-path shape in {path}: "
            f"{tuple(sampled_eval.shape)}"
        )
    path_rows.append(
        {
            "Stage": stage,
            "Run": spec.run_name,
            "Temperature": float(temperature),
            "Dense paths shape": str(tuple(sampled.shape)),
            "Evaluation paths shape": str(tuple(sampled_eval.shape)),
            "File": str(path),
        }
    )

path_manifest = pd.DataFrame(path_rows).drop_duplicates(
    subset=["Run", "Temperature"]
)
path_manifest.to_csv(
    EXPERIMENT_CONTROL_DIR / "sampled_path_manifest.csv",
    index=False,
)
display(path_manifest.style.set_caption("Saved ten-path decoded-price artefacts"))

final_manifest = {
    "project_git_commit": GIT_COMMIT,
    "output_root": str(OUTPUT_ROOT),
    "architecture_selection": architecture_selection,
    "temperature_selection": temperature_selection,
    "scale_source_run": ARCHITECTURE_WINNER_SPEC.run_name,
    "scale_run": SCALE_SPEC.run_name,
    "final_run": FINAL_SPEC.run_name,
    "final_run_dir": str(FINAL_RUN_DIR),
    "final_token_input": FINAL_SPEC.token_input_representation,
    "final_graph": FINAL_SPEC.graph_type,
    "final_temporal_family": FINAL_SPEC.temporal_family,
    "final_st_blocks": int(FINAL_SPEC.num_st_blocks),
    "final_uses_close_scale_features": FINAL_USES_SCALE,
    "selected_temperature": SELECTED_TEMPERATURE,
    "sample_count": SAMPLE_COUNT,
    "top_p": TOP_P,
    "top_k": TOP_K,
    "sampling_seed": SAMPLING_SEED,
    "final_mean_log_mae": FINAL_MEAN_LOG_MAE,
    "validation_split": "September 2024",
    "held_out_test_used": False,
    "final_sampled_path_file": str(
        scale_comparison.iloc[0]["Sampled path file"]
    ),
}
(EXPERIMENT_CONTROL_DIR / "final_selection.json").write_text(
    json.dumps(final_manifest, indent=2, sort_keys=True),
    encoding="utf-8",
)

print("\nFINAL MANIFEST")
print(json.dumps(final_manifest, indent=2, sort_keys=True))
